# Contratos de arrendamiento de ALeaseBERT: cuántos hay y cuántos tienen red flags

El dataset viene del paper de Leivaditi, Rossi y Kanoulas (2020), *A Benchmark for Lease Contract
Review* (arXiv 2010.10386; datos en Figshare, DOI 10.21942/uva.19732993). Son contratos de
arrendamiento comerciales sacados de SEC EDGAR y anotados con TagTog por una sola anotadora.

El dataset vive fuera del repositorio, en `../datasets/annotated_dataset/pool/`. Cada contrato es un
`.plain.html` (el texto, una línea por `<p>`) y, si se anotó, un `.ann.json` con el mismo nombre.
En las anotaciones, las red flags son la clase `e_23` y su tipo va en el campo `f_24`.

Este cuaderno regenera las cifras que usamos para elegir los contratos que convierte
`pool_to_docx.py`, y las compara con lo que dice el paper. No llama a ninguna API.

In [1]:
import collections
import json
from pathlib import Path

# La raíz del repositorio: la primera carpeta hacia arriba que contiene infra/.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'infra').is_dir():
    ROOT = ROOT.parent
DATASET_DIR = ROOT.parent / 'datasets' / 'annotated_dataset'
POOL = DATASET_DIR / 'pool'
RED_FLAG_CLASS, TYPE_FIELD = 'e_23', 'f_24'

# La leyenda de clases (annotations-legend.json, en Figshare) no viene en el zip del dataset.
# Si está junto a pool/, se usan sus nombres; si no, las clases se muestran por su id.
legend_path = DATASET_DIR / 'annotations-legend.json'
LEGEND = json.loads(legend_path.read_text()) if legend_path.exists() else {}

def label(class_id):
    return f"{LEGEND[class_id]} ({class_id})" if class_id in LEGEND else class_id

def name_of(path):
    """'aKWzND...-lease_contract_74.ann.json' -> 'lease_contract_74'"""
    return path.name.split('-', 1)[1].split('.')[0]

texts = {name_of(p): p for p in POOL.glob('*.plain.html')}
annotations = {name_of(p): json.loads(p.read_text()) for p in POOL.glob('*.ann.json')}

def red_flags(doc):
    return [e for e in doc['entities'] if e['classId'] == RED_FLAG_CLASS]

print('carpeta:', POOL)
print('leyenda de clases:', 'sí' if LEGEND else 'no está; las clases se muestran por id')

carpeta: /home/sante/Documents/FGV/me/datasets/annotated_dataset/pool
leyenda de clases: sí


## 1. Del total a los contratos con red flags

Cada paso se queda con una parte del anterior: tener texto, tener archivo de anotaciones, que ese
archivo traiga alguna anotación y que alguna de ellas sea una red flag. Los contratos del último
paso son los que `pool_to_docx.py` convierte a DOCX.

In [2]:
with_ann_file = [n for n in texts if n in annotations]
with_any = [n for n in with_ann_file if annotations[n]['entities']]
with_red_flags = [n for n in with_any if red_flags(annotations[n])]

steps = [
    ('contratos con texto (.plain.html)', len(texts)),
    ('con archivo de anotaciones (.ann.json)', len(with_ann_file)),
    ('con al menos una anotación', len(with_any)),
    ('con al menos una red flag', len(with_red_flags)),
]
for text, n in steps:
    print(f'{n:5d}  {text}')
print()
print('archivos de anotaciones sin texto:', len(set(annotations) - set(texts)))
print('red flags en total:', sum(len(red_flags(annotations[n])) for n in with_red_flags))

  335  contratos con texto (.plain.html)
  179  con archivo de anotaciones (.ann.json)
  123  con al menos una anotación
  115  con al menos una red flag

archivos de anotaciones sin texto: 0
red flags en total: 1252


## 2. Los archivos de anotaciones vacíos

Tener `.ann.json` no significa estar anotado: hay archivos sin ninguna anotación, y la mayoría
están marcados como terminados (`anncomplete`). Lo único que traen es la etiqueta de documento
`m_22`, el tipo de contrato, que el paper dice haber anotado de forma inconsistente.

In [3]:
empty = [n for n in with_ann_file if not annotations[n]['entities']]
print('archivos vacíos:', len(empty))
print('de ellos, marcados como terminados:', sum(annotations[n]['anncomplete'] for n in empty))
kinds = collections.Counter(annotations[n]['metas'].get('m_22', {}).get('value', '(sin tipo)') for n in empty)
print('tipo de contrato que traen:', dict(kinds))

archivos vacíos: 56
de ellos, marcados como terminados: 52
tipo de contrato que traen: {'sublease agreement': 5, 'amendment': 7, 'other': 42, 'lease agreement': 2}


## 3. Los que tienen anotaciones pero ninguna red flag

Quedan fuera de la conversión porque lo que se anotó en ellos son otras cosas (partes, fechas,
condiciones de pago, numeración de cláusulas…), no cláusulas de riesgo.

In [4]:
without_red_flags = [n for n in with_any if not red_flags(annotations[n])]
print('contratos:', len(without_red_flags))
for n in sorted(without_red_flags):
    classes = collections.Counter(e['classId'] for e in annotations[n]['entities'])
    print(f'\n{n}')
    for class_id, count in classes.most_common():
        print(f'    {count:3d}  {label(class_id)}')

contratos: 8

lease_476
      6  term_of_payment (e_9)
      5  clause_title (e_15)
      5  sub_clause_number (e_18)
      2  leased_space (e_3)
      1  signing_date (e_6)
      1  lessor (e_1)
      1  lessee (e_2)
      1  end_date (e_26)
      1  extension_period (e_13)
      1  sub_clause_title (e_17)
      1  annex (e_28)

lease_566
      6  sub_clause_number (e_18)
      3  clause_title (e_15)
      3  term_of_payment (e_9)
      1  lessor (e_1)
      1  lessee (e_2)
      1  leased_space (e_3)
      1  start_date (e_25)
      1  end_date (e_26)
      1  indexation_rent (e_10)

lease_570
     12  sub_clause_number (e_18)
      8  clause_number (e_16)
      8  clause_title (e_15)
      6  leased_space (e_3)
      1  signing_date (e_6)
      1  lessor (e_1)
      1  lessee (e_2)
      1  start_date (e_25)
      1  end_date (e_26)
      1  term_of_payment (e_9)
      1  annex (e_28)

lease_605
     23  sub_clause_number (e_18)
     16  definition_number (e_20)
      8  clause_numb

## 4. Red flags por tipo, frente a la Figura 1 del paper

La Figura 1 (derecha) del paper da un recuento por tipo de red flag. Aquí se copian sus valores tal
como aparecen y se ponen junto a dos recuentos de los datos publicados: cuántas anotaciones hay de
cada tipo y en cuántos contratos aparece. Los nombres de tipo son los del propio dataset, con sus
erratas (`compalsory_reconstraction`) y alguno en neerlandés.

In [5]:
PAPER_FIGURE_1 = {
    'services_charges': 145, 'sublease_permitted': 118, 'break_option': 114,
    'reinstatement_clause': 110, 'landlord_repairs': 84, 'damage': 68,
    'guarantee_transferable': 56, 'change_of_control': 47,
    'right_of_first_refusal_to_lease': 39, 'right_of_first_refusal_to_purchase': 26,
    'warrantees_of_the_owner': 21, 'compalsory_reconstraction': 10, 'expansion': 8,
    'special_stipulations': 6, 'extension_period': 5, 'no_obligation_to_operate': 1,
    'assignment_indeplaatsstelling_permitted': 1,
}

per_annotation, per_contract = collections.Counter(), collections.Counter()
for n in with_red_flags:
    kinds = [e['fields'].get(TYPE_FIELD, {}).get('value', '(sin tipo)') for e in red_flags(annotations[n])]
    per_annotation.update(kinds)
    per_contract.update(set(kinds))

print(f"{'tipo de red flag':42s} {'anotaciones':>11s} {'contratos':>9s} {'paper':>6s}")
for kind in sorted(set(per_annotation) | set(PAPER_FIGURE_1), key=lambda k: -per_annotation[k]):
    paper = PAPER_FIGURE_1.get(kind, '—')
    print(f'{kind:42s} {per_annotation[kind]:11d} {per_contract[kind]:9d} {paper:>6}')

same = [k for k in PAPER_FIGURE_1 if PAPER_FIGURE_1[k] in (per_annotation[k], per_contract[k])]
print()
print('tipos en los que la cifra del paper coincide con alguno de los dos recuentos:', len(same), 'de', len(PAPER_FIGURE_1))
print('tipos del dataset que el paper no describe:', sorted(set(per_annotation) - set(PAPER_FIGURE_1)))

tipo de red flag                           anotaciones contratos  paper
break_option                                       307        63    114
services_charges                                   190        93    145
sublease_permitted                                 169        83    118
reinstatement_clause                               130        71    110
landlord_repairs                                   101        59     84
damage                                              83        49     68
guarantee_transferable                              74        47     56
change_of_control                                   51        40     47
right_of_first_refusal_to_lease                     41        35     39
warrantees_of_the_owner                             27        11     21
right_of_first_refusal_to_purchase                  26        23     26
expansion                                           15         6      8
compalsory_reconstraction                           10         9

## Qué sacamos de esto

- Los contratos útiles para trabajar con red flags son los del paso 4 de la sección 1, no los
  179 que el paper presenta como anotados: el paper dice que todos tienen red flags.
- La Figura 1 solo cuadra en algunos tipos, los menos frecuentes. En los más frecuentes la cifra del
  paper no es ni el número de anotaciones ni el de contratos. Además hay red flags sin tipo y tipos
  que el paper no menciona. Con una sola anotadora y sin medida de acuerdo, conviene tratar estas
  anotaciones como una referencia débil, no como verdad de terreno.
- Una red flag marca una cláusula de riesgo para alguna de las partes. No dice a cuál favorece ni
  señala contradicciones.